# Biotic Hardware Synthesis — Comparative Morphology Demo v1.2

Pipeline completo: topology validation, Schumann baseline, 3-metric statistics, multi-seed distributions.

**Ejecutar desde la raíz del repo** (`biotic-hardware/`), no desde dentro de `notebook/`.

## 1 — Run the full pipeline

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, 'run.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 2 — Schumann resonance external baseline

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from data.schumann_reference import SCHUMANN_MODES_HZ, SCHUMANN_SOURCE, nearest_schumann_mode

with open('data/resonance_params.json') as f:
    res = json.load(f)

f_sim = res['f_resonance_Hz']
nearest_hz, mode_n, deviation = nearest_schumann_mode(f_sim)

fig, ax = plt.subplots(figsize=(10, 3))
for i, f in enumerate(SCHUMANN_MODES_HZ):
    color = '#FF5722' if i + 1 == mode_n else '#BDBDBD'
    label = f'Mode {i+1} ({f} Hz)' if i + 1 == mode_n else f'Mode {i+1}'
    ax.axvline(f, color=color, linewidth=2, label=label)
ax.axvline(f_sim, color='#2196F3', linewidth=2.5, linestyle='--',
           label=f'Simulated ({f_sim:.4f} Hz)')
ax.set_xlabel('Frequency (Hz)')
ax.set_title(
    f'Schumann Resonance Comparison — deviation {deviation:.2f}% from mode {mode_n}'
    f' | {SCHUMANN_SOURCE}'
)
ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_xlim(0, 40)
plt.tight_layout()
plt.show()

print(f'Simulated : {f_sim:.4f} Hz')
print(f'Nearest   : {nearest_hz:.2f} Hz  (mode {mode_n})')
print(f'Deviation : {deviation:.2f}%')

## 3 — Morphological sensitivity curves (normalized)

In [ ]:
import pandas as pd

df_f = pd.read_csv('data/simulation_results_fractal.csv')
df_b = pd.read_csv('data/simulation_results_botanical.csv')
df_r = pd.read_csv('data/simulation_results_random.csv')

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_f['Distance'], norm(df_f['Merit_Scaled']),    label='Fractal - Merit Scaled',
        linewidth=2, color='#2196F3')
ax.plot(df_f['Distance'], norm(df_f['Coherence_Ratio']), label='Fractal - Coherence',
        linewidth=1.2, color='#2196F3', linestyle='--', alpha=0.7)
ax.plot(df_b['Distance'], norm(df_b['Merit_Scaled']),    label='Botanical - Merit Scaled',
        linewidth=2, color='#4CAF50')
ax.plot(df_b['Distance'], norm(df_b['Coherence_Ratio']), label='Botanical - Coherence',
        linewidth=1.2, color='#4CAF50', linestyle='--', alpha=0.7)
ax.plot(df_r['Distance'], norm(df_r['Merit_Scaled']),    label='Random - Merit Scaled',
        linewidth=2, color='#FF5722')
ax.plot(df_r['Distance'], norm(df_r['Coherence_Ratio']), label='Random - Coherence',
        linewidth=1.2, color='#FF5722', linestyle='--', alpha=0.7)

ax.set_xlabel('Distance (m)')
ax.set_ylabel('Normalized value')
ax.set_title('Morphological Sensitivity Benchmark v1.2 — Fractal / Botanical / Random Control (seed 42)')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4 — Statistical separation: 3 metrics x 3 pairs

Welch t-test + Cohen's d leídos desde `data/statistical_summary.csv` (generado por el pipeline).  
**p-value**: verde = significativo. **|Cohen's d|**: azul = efecto grande.

In [ ]:
df_stat = pd.read_csv('data/statistical_summary.csv')

metrics = ['Merit_Scaled', 'Coherence_Ratio', 'Peak_AF']
pairs   = ['Fractal vs Botanical', 'Fractal vs Random', 'Botanical vs Random']

p_matrix = np.zeros((len(metrics), len(pairs)))
d_matrix = np.zeros((len(metrics), len(pairs)))

for _, row in df_stat.iterrows():
    i = metrics.index(row['Metric'])
    j = pairs.index(row['Pair'])
    p_matrix[i, j] = row['p_value']
    d_matrix[i, j] = abs(row['Cohens_d'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

im1 = ax1.imshow(p_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.1, aspect='auto')
ax1.set_xticks(range(len(pairs)))
ax1.set_xticklabels(pairs, rotation=20, ha='right', fontsize=9)
ax1.set_yticks(range(len(metrics)))
ax1.set_yticklabels(metrics)
ax1.set_title('p-value (verde = significativo, umbral 0.05)')
for i in range(len(metrics)):
    for j in range(len(pairs)):
        txt_color = 'white' if p_matrix[i, j] < 0.01 else 'black'
        ax1.text(j, i, f'{p_matrix[i,j]:.4f}', ha='center', va='center',
                 fontsize=9, color=txt_color)
plt.colorbar(im1, ax=ax1)

d_max = max(d_matrix.max(), 1.0)
im2 = ax2.imshow(d_matrix, cmap='Blues', vmin=0, vmax=d_max, aspect='auto')
ax2.set_xticks(range(len(pairs)))
ax2.set_xticklabels(pairs, rotation=20, ha='right', fontsize=9)
ax2.set_yticks(range(len(metrics)))
ax2.set_yticklabels(metrics)
ax2.set_title("|Cohen's d| (azul oscuro = efecto grande >0.8)")
for i in range(len(metrics)):
    for j in range(len(pairs)):
        txt_color = 'white' if d_matrix[i, j] > d_max * 0.6 else 'black'
        ax2.text(j, i, f'{d_matrix[i,j]:.2f}', ha='center', va='center',
                 fontsize=9, color=txt_color)
plt.colorbar(im2, ax=ax2)

fig.suptitle('Statistical Separation — Welch t-test + Cohen d (v1.2)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

df_stat[['Metric', 'Pair', 'p_value', 'Significant_p05', 'Cohens_d', 'Effect_size']]

## 5 — Multi-seed analysis: distribuciones por morfología (seeds 42-46)

Cada barra es la media sobre 5 seeds. Barras de error = ±1 std.  
Fractal std bajo → estable. Botanical std alto → sensible a geometría de seed.

In [ ]:
df_multi = pd.read_csv('data/multi_seed_summary.csv')

morphologies = df_multi['Morphology'].tolist()
colors = {'fractal': '#2196F3', 'botanical': '#4CAF50', 'random': '#FF5722'}

metrics_multi = [
    ('Merit_Mean',     'Merit_Std',     'Merit Scaled'),
    ('Coherence_Mean', 'Coherence_Std', 'Coherence Ratio'),
    ('PeakAF_Mean',    'PeakAF_Std',    'Peak AF'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (mean_col, std_col, label) in zip(axes, metrics_multi):
    means = df_multi[mean_col].values
    stds  = df_multi[std_col].values
    bar_colors = [colors.get(m, '#888888') for m in morphologies]
    bars = ax.bar(morphologies, means, yerr=stds, color=bar_colors,
                  capsize=6, alpha=0.85, width=0.5)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + max(means) * 0.03,
            f'{mean:.4f}\n+/-{std:.4f}',
            ha='center', va='bottom', fontsize=8
        )
    ax.set_title(label)
    ax.set_ylabel('Value')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Multi-seed Sensitivity Analysis — seeds [42, 43, 44, 45, 46] (v1.2)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — exploration_summary.json

Output machine-readable: configuración experimental + baseline Schumann + resultados multi-seed en un único punto de entrada estructurado.

In [ ]:
with open('data/exploration_summary.json') as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))